# BARC-SS example: HUC8 11110203, trapezoidal channel with a = 0.4

This notebook adjusts the OWP HAND-FIM rating curves of HUC8 **11110203** (Arkansas River basin, Arkansas) with **BARC-SS**, then maps the flood of **3 January 2016** from the NWM retrospective discharge.

Steps:
1. Download the OWP HAND-FIM outputs for the HUC8 with [FIMserv](https://github.com/sdmlua/FIMserv).
2. Get bankfull and mean-flow channel geometry for every reach from [HydroGeom](https://github.com/Reizrb/HydroGeom), and compute bankfull area and wetted perimeter for a trapezoidal channel (a = 0.4).
3. Apply BARC-SS to the rating curves.
4. Compare a rating curve before and after.
5. Get NWM discharge for the event and run HAND-FIM.
6. Map the inundated area.

**Run this notebook from an empty folder.** FIMserv creates `code/`, `data/`, and `output/` folders where it runs. The HUC8 download is a few GB.

## Setup

Install once (in a terminal, or uncomment the line below):

```bash
pip install "barcss[example] @ git+https://github.com/Reizrb/Bathymetric-Adjustment-of-Rating-Curves-and-Stage-Shift-BARC-SS"
```

In [ ]:
# !pip install "barcss[example] @ git+https://github.com/Reizrb/Bathymetric-Adjustment-of-Rating-Curves-and-Stage-Shift-BARC-SS"
import fimserve as fm
import matplotlib.pyplot as plt
import barcss

## Settings

- `method`: `"BARC-SS"` (geometry + stage shift by mean-flow depth) or `"BARC"` (geometry only)
- `shape` and its parameter: the channel shape used to turn bankfull width and depth into area and wetted perimeter
  - `shape="a", a=0.4`: trapezoid whose bottom width is 0.4 × top width (`a=0` triangle, `a=1` rectangle)
  - `shape="r", r=2`: Dingman's shape parameter (`r=2` parabola)
  - `shape="z", z=2`: side slope 2:1

In [ ]:
huc = "11110203"
method = "BARC-SS"
shape = "a"
shape_param = {"a": 0.4}

# Flood event
start_date = "2015-12-26"
end_date = "2016-01-05"
value_times = ["2016-01-03 00:00:00"]

output_dir = "output"   # FIMserv's output folder

## Step 1. Download the OWP HAND-FIM outputs (version 4.5)

In [ ]:
fm.DownloadHUC8(huc, version="4.5")

## Step 2. Channel geometry for every reach

`feature_ids` lists the NWM reaches (NHDPlusV2.1 COMIDs) in this HUC8's rating curves. `bankfull_geometry` gets their predicted bankfull and mean-flow width and depth from HydroGeom (Zarrabi et al., 2025) and computes the bankfull cross-sectional area (`bnk_xsce_A`) and wetted perimeter (`bnk_wet_p`) for the chosen shape. The mean-flow depth (`mf_depth`) is the BARC-SS stage shift.

In [ ]:
ids = barcss.feature_ids(output_dir, huc)
geometry = barcss.bankfull_geometry(ids, shape=shape, **shape_param)
print(f"{len(ids):,} reaches in the rating curves, {geometry['bnk_xsce_A'].notna().sum():,} with predicted geometry")
geometry.head()

## Step 3. Apply BARC-SS

This rewrites each branch's `src_full_crosswalked_*.csv` and `hydroTable_*.csv`. The original files are saved in `output/flood_11110203/barcss_original/` the first time, and every run starts from them, so you can run again with another method or shape. `barcss.restore(output_dir, huc)` puts the originals back.

In [ ]:
hydrotable = barcss.apply(output_dir, huc, geometry, method=method)

## Step 4. Rating curve before and after

In [ ]:
barcss.plot.rating_curve(output_dir, huc)   # or pass hydroid=... for a specific reach
plt.show()

## Step 5. NWM discharge for the event, then run HAND-FIM

In [ ]:
fm.getNWMretrospectivedata(huc, start_date, end_date, value_times=value_times)
fm.runOWPHANDFIM(huc)

## Step 6. Flood map

In [ ]:
label = f"{method}_{shape}{list(shape_param.values())[0]}"
barcss.plot.flood_map(output_dir, huc)
plt.savefig(f"flood_map_{huc}_{label}.png", dpi=200, bbox_inches="tight")
plt.show()

## Try another method or channel shape

FIMserv skips a run if a flood map for the same discharge already exists, so keep the current map under its own name first, then change the settings and run Steps 2 to 6 again (Step 1 is not needed).

In [ ]:
barcss.keep_flood_map(output_dir, huc, label)

# For example, BARC with a rectangular channel:
# method, shape, shape_param = "BARC", "a", {"a": 1.0}

To map the original HAND-FIM (no adjustment) for comparison:

```python
barcss.restore(output_dir, huc)
fm.runOWPHANDFIM(huc)
barcss.keep_flood_map(output_dir, huc, "original")
```

A benchmark flood map for this event (Tier 1, NOAA aerial imagery, 2016-01-03) is available in [FIMbench](https://fimbench.tethys.ciroh.org/) to evaluate the maps.